In [ ]:
import os

os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"


import numpy as np
import pandas as pd
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import os
import scanpy as sc
import plotnine as gg

import matplotlib.pyplot as plt
import plotly.express as px
import matplotlib.colors as mcolors
from tqdm import tqdm


tab10_colors = plt.get_cmap("tab10").colors
tab10_hex = [mcolors.to_hex(c) for c in tab10_colors]
plt.rcParams["svg.fonttype"] = "none"

In [ ]:
!pip install pyarrow

In [ ]:
f_vector_cols = [
    # "N Mismatch_y",
    "Delta time (s)",
    "Instantaneous Growth Rate: Volume",
    "Length",
    "Septum Displacement Length Normalized",
    "Width",
    "mCherry mean_intensity",
]


timeseries_df = pd.read_pickle(
    "/workspace/data/Eaton_2025/Data/lDE20_Imaging/Clustering/2023-01-23_sgRNA_Timeseries_df.pkl"
)
working_dir = "/workspace/data/Eaton_2025/Data/lDE20_Imaging"
df = pd.read_csv(
    os.path.join(working_dir, "2024-01-25_lDE20_Steady_State_df_Estimators_wStats.csv")
)

df_ = df.query("Estimator == 'Mean (Robust)'")
ops_summarystats_df = (
    df_.pivot(index=["sgRNA", "Gene", "N Mismatch"], columns="Variable(s)", values="Value")
    .reset_index()
    .set_index(["sgRNA", "Gene"])
)
ops_summarystats_df


def expand_embeddings(df, columns):
    expanded_dfs = []
    for col in columns:
        if col not in df.columns:
            continue
        col_values = np.stack(df[col].values)
        if col_values.ndim > 2:
            col_values = col_values.reshape(len(df), -1)

        expanded = pd.DataFrame(
            col_values, index=df.index, columns=[f"{col}_{i}" for i in range(col_values.shape[1])]
        )
        expanded_dfs.append(expanded)

    metadata_cols = [c for c in df.columns if c not in columns]
    return pd.concat([df[metadata_cols]] + expanded_dfs, axis=1).reset_index()


df_embeddings = expand_embeddings(timeseries_df, ["Feature Vector"])
tsne_rep = TSNE(n_components=2)
embeddings_tsne = tsne_rep.fit_transform(
    df_embeddings[[c for c in df_embeddings.columns if c.startswith("Feature Vector")]]
)
df_embeddings["tsne_x"] = embeddings_tsne[:, 0]
df_embeddings["tsne_y"] = embeddings_tsne[:, 1]
df_embeddings.info()

pca_ = PCA(n_components=2)
pca_.fit(df_embeddings[[c for c in df_embeddings.columns if c.startswith("Feature Vector")]])
embeddings_pca = pca_.transform(
    df_embeddings[[c for c in df_embeddings.columns if c.startswith("Feature Vector")]]
)
df_embeddings["pca_x"] = embeddings_pca[:, 0]
df_embeddings["pca_y"] = embeddings_pca[:, 1]
# comparison with growth fitness screen

ops_df = df_embeddings.merge(ops_summarystats_df.reset_index(), on="sgRNA", how="left")
# ops_df

In [ ]:
adata = sc.read_h5ad(
    "/workspace/data/251117_genomescale_CRISPRi/sample_mix_umi200_hvg500_pc25_neighbors10_mindist0.55.scvi.h5ad"
)
adata.X = adata.layers["reads"].copy()
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

adata.obs["transcript_UMAP1"] = adata.obsm["X_umap"][:, 0]
adata.obs["transcript_UMAP2"] = adata.obsm["X_umap"][:, 1]